# Valora AI — Qwen3 4B Fine-Tuning v2.0 (Production-Aligned)

**GIS Reasoning Agent for Bangalore Real Estate Intelligence**

## Critical Alignment (v2.0 fixes)
| What | v1.0 (WRONG) | v2.0 (FIXED) |
|------|-------------|-------------|
| **Facts format** | `[GROUNDED FACTS]` with `###` headers | `**bold**` headers + `  -` bullets (matches `to_context_string()`) |
| **System prompt** | Condensed 8-rule custom prompt | Condensed constitution + intent instruction (matches production pattern) |
| **LLM call** | Chat template (but production used raw text) | Chat format — production NOW uses `/api/chat` properly |
| **Facts header** | `[GROUNDED FACTS]` | `**GROUNDED FACTS (use ONLY these):**` (matches `_build_llm_messages()`) |
| **Intents** | 12 custom categories | 13 production intents (added investment, recommendation) |
| **Production fields** | Basic (price, walk, metro) | ALL fields (archetype, risk_profile, causal, livability, etc.) |

## Dataset v2.0 (3,600 examples)
| Metric | Value |
|--------|-------|
| **Total** | 3,600 |
| **Train** | 3,240 (90%) |
| **Eval** | 360 (10%) |
| **Intents** | 13 production intents |
| **Localities** | 30 Bangalore areas |

### Intent Distribution
| Intent | Count | Purpose |
|--------|-------|---------|
| analyze_area | 500 | Comprehensive area analysis |
| property_search | 500 | Find listings by criteria |
| comparison | 350 | Side-by-side locality comparison |
| investment | 350 | ROI, rental yield, entry strategy |
| simulate | 300 | What-if infrastructure scenarios |
| valuation | 300 | Fair market value estimation |
| navigate | 250 | 3D map navigation with context |
| analyze_building | 250 | 3D shadow, view, floor analysis |
| market_trend | 250 | Price trajectory, supply-demand |
| terrain | 200 | Elevation, flood risk, drainage |
| recommendation | 150 | Personalized area suggestions |
| general | 100 | Capabilities, limitations, redirect |
| conversational | 100 | Greetings, thanks, farewell |

## Training Architecture (Best Quality for 4B)
- **Base Model**: Qwen3-4B (Qwen/Qwen3-4B)
- **Method**: QLoRA 4-bit with LoRA rank **128**, alpha **256**
- **Context**: **4096 tokens** (system ~600tok + facts ~300tok + response ~400tok fits well)
- **Epochs**: **4** (3.6K dataset benefits from extra passes)
- **LR**: **1e-5** (slow, careful learning)
- **Effective Batch**: **16** (batch 2 × grad_accum 8)
- **Packing**: **ON** (fills context window efficiently — ~3 examples/sequence)
- **rsLoRA**: **ON** (rank-stabilized for r=128)
- **Template**: Qwen3 native ChatML via `apply_chat_template`

### Production Pipeline Alignment
```
Training:  system(constitution + intent + facts) → user(query) → assistant(<think> + response)
    ↕ IDENTICAL FORMAT ↕
Production: _build_llm_messages() → /api/chat → Ollama native ChatML template
```

In [ ]:
# Cell 1: Configuration (v2.0 Production-Aligned)
import os
from pathlib import Path

print("=" * 60)
print("  VALORA AI — QWEN3 4B FINE-TUNING v2.0")
print("  Production-Aligned: /api/chat + to_context_string()")
print("=" * 60)

# ===== MODEL =====
MODEL_NAME = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 2048              # Reduced for 14GB VRAM
LORA_RANK = 64                     # Reduced from 128 to save optimizer VRAM
LORA_ALPHA = 128                   # Keep 2× rank ratio

# ===== TRAINING (Optimized for T4 14GB VRAM) =====
NUM_EPOCHS = 4
LEARNING_RATE = 1e-5
BATCH_SIZE = 2                    # Increased to use more VRAM (was 1)
GRAD_ACCUM = 8                    # Reduced to keep effective batch = 16
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.3
USE_RSLORA = True
LORA_DROPOUT = 0.0
USE_PACKING = True

# ===== PATHS =====
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("/kaggle/working/final_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = Path("/kaggle/input/valora-training-data")
TRAIN_FILE = PROJECT_ROOT / "train.json"
EVAL_FILE = PROJECT_ROOT / "eval.json"

# ===== RESUME =====
def find_latest_checkpoint():
    checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"),
                        key=lambda x: int(x.name.split("-")[1]) if x.name.split("-")[1].isdigit() else 0)
    return str(checkpoints[-1]) if checkpoints else None

latest_ckpt = find_latest_checkpoint()

print(f"Model: {MODEL_NAME}")
print(f"Context: {MAX_SEQ_LENGTH} | LoRA: r={LORA_RANK}, α={LORA_ALPHA}, rsLoRA={USE_RSLORA}")
print(f"Training: {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, batch={BATCH_SIZE}×{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}")
print(f"Packing: {USE_PACKING} | Dropout: {LORA_DROPOUT}")
print(f"Dataset: v2.0 (production-aligned, /api/chat format)")
if latest_ckpt:
    print(f"RESUME FROM: {Path(latest_ckpt).name}")
else:
    print("Starting fresh")
print("=" * 60)

In [ ]:
# Cell 2: Progress Tracker
import threading, time, json
from datetime import datetime

class ProgressTracker:
    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.progress_file = self.checkpoint_dir / "progress.json"
        self.start_time = time.time()
        self.running = False
        self.thread = None
        self.step = 0
        self.total = 0
        self.loss = None

    def _log(self):
        while self.running:
            elapsed = time.time() - self.start_time
            h, m = int(elapsed // 3600), int((elapsed % 3600) // 60)
            s = f"⏱️ {h:02d}:{m:02d}"
            if self.total > 0:
                s += f" | {self.step}/{self.total} ({self.step/self.total*100:.1f}%)"
            if self.loss: s += f" | Loss: {self.loss:.4f}"
            print(f"\r{s}", end='', flush=True)
            time.sleep(60)

    def start(self, total=0):
        self.total = total; self.running = True
        self.thread = threading.Thread(target=self._log, daemon=True)
        self.thread.start()

    def update(self, step, loss=None):
        self.step = step
        if loss: self.loss = loss
        with open(self.progress_file, 'w') as f:
            json.dump({"step": step, "total": self.total, "loss": self.loss,
                       "elapsed": time.time()-self.start_time,
                       "timestamp": datetime.now().isoformat()}, f, indent=2)

    def stop(self):
        self.running = False
        if self.thread: self.thread.join(timeout=2)

tracker = ProgressTracker(CHECKPOINT_DIR)
prev = None
if tracker.progress_file.exists():
    with open(tracker.progress_file) as f: prev = json.load(f)
    print(f"Previous: Step {prev['step']}, Loss {prev.get('loss', 'N/A')}")
else:
    print("No previous progress")

In [ ]:
# Cell 3: Install Dependencies (Kaggle-optimized)
# CRITICAL: --no-deps on unsloth to prevent torch 2.8→2.10 upgrade
# Pin trl<0.24 to avoid SamplingParams/vllm error
# Install peft from git source as recommended by Unsloth error message
print("📦 Installing dependencies...")

# 1. Install peft from git (Unsloth's recommendation)
!pip install git+https://github.com/huggingface/peft.git -q

# 2. Training libraries (safe, won't touch torch)
!pip install bitsandbytes accelerate -q
!pip install "trl>=0.18.2,<0.24.0" -q
!pip install sentencepiece protobuf cut_cross_entropy msgspec tyro hf_transfer -q

# 3. Pin datasets<4.4 (unsloth breaks with >=4.4.0)
!pip install "datasets>=3.4.1,<4.4.0" -q

# 4. Install unsloth + zoo (--no-deps prevents torch upgrade)
import gc; gc.collect()
!pip install "unsloth_zoo==2026.1.4" --no-deps -q
!pip install "unsloth==2026.1.4" --no-deps -q

# 5. Import unsloth FIRST (required for optimizations)
import unsloth

# Set HuggingFace timeout environment variables
import os
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['REQUESTS_TIMEOUT'] = '60'

# Set memory optimization for T4
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Auto-detect GPU and set optimizations accordingly
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_arch = torch.cuda.get_device_capability(0)[0]  # sm_XX
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"✅ GPU: {gpu_name} (sm_{gpu_arch}0) - {gpu_mem:.1f} GB VRAM")
    
    if gpu_arch >= 70:  # sm_70+ (Turing/Ampere) - enable all optimizations
        print("  GPU supports Triton optimizations - enabling all Unsloth features")
        # No need to disable anything, use defaults
    else:  # sm_60 (P100) - disable Triton to avoid PTXAS errors
        print("  Legacy GPU detected - disabling Triton for compatibility")
        os.environ['UNSLOOTH_DISABLE_COMPILATION'] = '1'  # Disable torch.compile
        os.environ['UNSLOOTH_DISABLE_TRITON'] = '1'      # Disable Triton kernels
        os.environ['UNSLOOTH_DISABLE_FAST_KERNELS'] = '1' # Disable fast kernels
else:
    raise RuntimeError("GPU required for training")

# Verify versions
import peft
print(f"  torch: {torch.__version__}")
print(f"  peft: {peft.__version__}")
assert "2.8" in torch.__version__, f"torch upgraded to {torch.__version__}! Restart kernel."

# Verify peft supports rsLoRA
try:
    from peft import LoraConfig
    sig = LoraConfig.__init__.__code__.co_varnames
    has_rslora = 'use_rslora' in sig
    print(f"  peft supports use_rslora: {has_rslora}")
    assert has_rslora, f"peft {peft.__version__} missing use_rslora!"
except Exception as e:
    print(f"  peft check failed: {e}")
    raise

print(f"  Memory optimization: expandable_segments=True")

In [ ]:
# Cell 4: Load Qwen3 4B Model
from unsloth import FastLanguageModel
import torch, time

print(f"Loading: {MODEL_NAME}")

max_retries = 3
for attempt in range(max_retries):
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=True,
        )
        break
    except Exception as e:
        if attempt < max_retries - 1:
            wait = 30 * (attempt + 1)
            print(f"  Retry {attempt+1}/{max_retries} in {wait}s...")
            time.sleep(wait)
        else:
            raise

vram = torch.cuda.memory_allocated(0) / 1024**3
print(f"Loaded ({vram:.2f} GB VRAM)")
print(f"  Qwen3 4B — optimized for GIS reasoning with <think> tags")

In [ ]:
# Cell 5: Setup Chat Template (Qwen3 native)
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",  # Qwen3 uses same ChatML-style template
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
    map_eos_token=True,
)

print("Chat template configured (Qwen3 ChatML)")
print("  Supports <think>...</think> reasoning tags natively")

In [ ]:
# Cell 6: Add LoRA Adapters (Optimized for T4 14GB)
# Patch Unsloth's version check for peft (it doesn't recognize 0.18.2.dev0)
import peft
import torch

# Check what version Unsloth actually accepts
import re
version_pattern = re.compile(r'^\d+\.\d+\.\d+$')
if not version_pattern.match(peft.__version__):
    peft.__version__ = "0.9.0"

# Alternative: disable use_rslora temporarily if version check fails
try:
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,                    # 64 — good capacity, fits VRAM
        lora_alpha=LORA_ALPHA,          # 128 — 2× rank ratio
        lora_dropout=LORA_DROPOUT,      # 0.0 — Unsloth recommendation
        bias="none",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        random_state=3407,
        use_rslora=USE_RSLORA,
        use_gradient_checkpointing="unsloth",  # MUST be on for 14GB VRAM
    )
except RuntimeError as e:
    if "does not support `use_rslora`" in str(e):
        print("Falling back: disabling use_rslora due to version check")
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_RANK,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"
            ],
            random_state=3407,
            use_gradient_checkpointing="unsloth",
        )
    else:
        raise

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
vram_used = torch.cuda.memory_allocated(0) / 1024**3
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, rsLoRA={USE_RSLORA}")
print(f"  Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")
print(f"  Targets: attention (q/k/v/o) + MLP (gate/up/down)")
print(f"  Gradient checkpointing: ENABLED (unsloth)")
print(f"  VRAM after LoRA: {vram_used:.1f} GB")

In [ ]:
# Cell 7: Load Training Data
import json

print(f"Loading data from {PROJECT_ROOT}")

if not TRAIN_FILE.exists():
    print(f"Missing: {TRAIN_FILE}")
    print("  Attach 'valora-training-data' dataset!")
    raise FileNotFoundError(f"Missing: {TRAIN_FILE}")

with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    train_data = json.load(f)
with open(EVAL_FILE, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

print(f"Loaded: {len(train_data):,} train + {len(eval_data):,} eval")

# Verify v2.0 structure
sample = train_data[0]
roles = [m['role'] for m in sample['messages']]
has_think = '<think>' in str(sample)
has_facts = '**GROUNDED FACTS' in str(sample)
has_bold = '**Location:**' in str(sample) or '**Market Data:**' in str(sample)

print(f"  Roles: {roles}")
print(f"  Has <think>: {has_think}")
print(f"  Has **GROUNDED FACTS**: {has_facts}")
print(f"  Has **bold** headers: {has_bold}")

assert roles == ['system', 'user', 'assistant'], f"Bad roles: {roles}"
assert has_facts, "Missing GROUNDED FACTS header — wrong dataset version?"
print("  v2.0 format verified ✓")

In [ ]:
# Cell 8: Convert to Text Format
def convert_to_text(examples):
    formatted = []
    for ex in examples:
        msgs = ex.get("messages", [])
        if not msgs: continue
        conv = []
        for m in msgs:
            role = m.get("role", "")
            content = m.get("content", "")
            if isinstance(content, list):
                content = " ".join(item.get("text", "") for item in content if isinstance(item, dict))
            if content.strip():
                conv.append({"role": role, "content": content})
        if conv:
            formatted.append({"conversations": conv})
    return formatted

train_fmt = convert_to_text(train_data)
eval_fmt = convert_to_text(eval_data)
print(f"Converted: {len(train_fmt):,} train + {len(eval_fmt):,} eval")

In [ ]:
# Cell 9: Create HuggingFace Datasets
from datasets import Dataset

def create_dataset(formatted_data):
    records = []
    for item in formatted_data:
        text = tokenizer.apply_chat_template(
            item["conversations"],
            tokenize=False,
            add_generation_prompt=False
        )
        records.append({"text": text})
    return Dataset.from_list(records)

train_dataset = create_dataset(train_fmt)
eval_dataset = create_dataset(eval_fmt)

print(f"Datasets: {len(train_dataset):,} train | {len(eval_dataset):,} eval")
sample_texts = train_dataset.select(range(min(100, len(train_dataset))))['text']
avg_len = sum(len(t) for t in sample_texts) / len(sample_texts)
print(f"  Avg text length: {avg_len:.0f} chars (~{int(avg_len/4)} tokens)")

In [ ]:
# Cell 10: Training Config (Optimized for T4 14GB VRAM)
from trl import SFTConfig
from unsloth import is_bfloat16_supported
import torch

total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS

training_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),

    per_device_train_batch_size=BATCH_SIZE,      # 2 to use more VRAM
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,       # 8 (still effective batch = 16)

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    gradient_checkpointing=True,  # MUST be on for 14GB VRAM
    optim="adamw_8bit",
    max_grad_norm=MAX_GRAD_NORM,

    logging_steps=10,
    logging_first_step=True,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,

    eval_strategy="steps",
    eval_steps=100,

    resume_from_checkpoint=True,
    load_best_model_at_end=False,

    seed=3407,
    report_to="none",
    max_seq_length=MAX_SEQ_LENGTH,  # 2048
    packing=USE_PACKING,

    dataloader_num_workers=0,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

print(f"Config: {NUM_EPOCHS} epochs, ~{total_steps} steps")
print(f"  LR: {LEARNING_RATE} (cosine) | Batch: {BATCH_SIZE}×{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}")
print(f"  Context: {MAX_SEQ_LENGTH} | Packing: {USE_PACKING}")
print(f"  Gradient checkpointing: ENABLED")
print(f"  Using more VRAM: batch_size=2 (was 1)")

In [ ]:
# Cell 11: Initialize Trainer
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_config,
    dataset_text_field="text",
)

checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"),
                    key=lambda x: int(x.name.split("-")[1]) if x.name.split("-")[1].isdigit() else 0)
if checkpoints:
    print(f"Found {len(checkpoints)} checkpoint(s), latest: {checkpoints[-1].name}")
else:
    print("No checkpoints — starting fresh")
print(f"Dataset: {len(train_dataset):,} train + {len(eval_dataset):,} eval")

In [ ]:
# Cell 12: TRAIN
import gc, torch
from pathlib import Path

print("=" * 60)
print("  STARTING FINE-TUNING — QWEN3 4B")
print("=" * 60)

checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"),
                    key=lambda x: int(x.name.split("-")[1]) if x.name.split("-")[1].isdigit() else 0)
resume_from = str(checkpoints[-1]) if checkpoints else None

if resume_from:
    print(f"RESUMING FROM: {Path(resume_from).name}")
else:
    print("FRESH TRAINING")

# Clear memory before training
gc.collect()
torch.cuda.empty_cache()

# Show memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {allocated:.1f}GB allocated, {reserved:.1f}GB reserved, {total:.1f}GB total")

# Use the correct total steps from training config
tracker.start(total=total_steps)

try:
    result = trainer.train(resume_from_checkpoint=resume_from)

    print("\n" + "=" * 60)
    print("  TRAINING COMPLETE")
    print("=" * 60)
    print(f"  Final loss: {result.training_loss:.4f}")
    print(f"  Steps: {result.global_step}")

    print("\nSaving final model...")
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print(f"  Saved to: {OUTPUT_DIR}")
    tracker.stop()

except KeyboardInterrupt:
    print("\n  INTERRUPTED — re-run this cell to resume")
    checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
    if checkpoints: print(f"  Last checkpoint: {checkpoints[-1].name}")
    tracker.stop()

except Exception as e:
    print(f"\n  ERROR: {type(e).__name__}: {str(e)[:200]}")
    checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
    if checkpoints: print(f"  Last checkpoint: {checkpoints[-1].name}")
    tracker.stop()
    raise

In [ ]:
# Cell 13: Export to GGUF for Ollama
print("EXPORTING TO GGUF")
print("=" * 60)

if not (OUTPUT_DIR / "adapter_config.json").exists():
    print("No final model found. Run training first.")
    checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
    if checkpoints: print(f"  Latest checkpoint: {checkpoints[-1]}")
else:
    print("Exporting Q4_K_M quantization...")
    try:
        model.save_pretrained_gguf(
            str(OUTPUT_DIR / "gguf"),
            tokenizer,
            quantization_method="q4_k_m"
        )
        print("GGUF Q4_K_M export complete!")
    except Exception as e:
        print(f"Q4_K_M failed: {e}")

    print("\nExporting Q8_0 quantization...")
    try:
        model.save_pretrained_gguf(
            str(OUTPUT_DIR / "gguf_q8"),
            tokenizer,
            quantization_method="q8_0"
        )
        print("GGUF Q8_0 export complete!")
    except Exception as e:
        print(f"Q8_0 failed: {e}")
        print("  Export manually after downloading LoRA adapters")

In [ ]:
# Cell 14: Create Modelfile for Ollama
modelfile = '''FROM valora-qwen3-4b.gguf

TEMPLATE """{{- range .Messages }}
{{- if eq .Role "system" }}<|im_start|>system
{{ .Content }}<|im_end|>
{{ end }}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ end }}
{{- if eq .Role "assistant" }}<|im_start|>assistant
{{ .Content }}<|im_end|>
{{ end }}
{{- end }}<|im_start|>assistant
"""

PARAMETER stop <|im_end|>
PARAMETER stop <|im_start|>
PARAMETER temperature 0.6
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096
PARAMETER num_predict 1024

SYSTEM """You are Valora AI, a GIS reasoning agent for Bangalore real estate intelligence.

## RULES
1. GROUNDED FACTS ONLY — Every number must come from [GROUNDED FACTS]. Never invent data.
2. REASONING — Put internal reasoning in <think>...</think> tags. Final answer OUTSIDE tags.
3. BANGALORE ONLY — Decline non-Bangalore queries politely.
4. OFFLINE DATA — 686K buildings, 42K properties, 27K POIs from local database.
5. INDIAN FORMAT — Use ₹, lakhs/crores, sqft, BHK. Bold key metrics.
6. CONFIDENCE — End with: Confidence: HIGH/MEDIUM/LOW based on [reason].
7. CONCISE — Direct answer first, evidence, then actionable insight.
8. LIMITATIONS — If data missing, say so honestly."""
'''

with open(OUTPUT_DIR / "Modelfile.valora", 'w') as f:
    f.write(modelfile)

print("Modelfile.valora created")
print(f"  Location: {OUTPUT_DIR / 'Modelfile.valora'}")

In [ ]:
# Cell 15: Summary
print("=" * 60)
print("  VALORA AI — FINE-TUNING v2.0 COMPLETE")
print("  Production-Aligned: /api/chat + to_context_string()")
print("=" * 60)

print(f"\nMODEL: Qwen3 4B (GIS Reasoning Agent)")
print(f"DATASET: 3,600 examples v2.0 (production-aligned)")
print(f"INTENTS: 13 production intents")
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, rsLoRA={USE_RSLORA}")
print(f"Context: {MAX_SEQ_LENGTH} | Packing: {USE_PACKING}")

print(f"\nALIGNMENT VERIFIED:")
print("  ✓ Facts format: to_context_string() (**bold** + '  - ' bullets)")
print("  ✓ Facts header: '**GROUNDED FACTS (use ONLY these):**'")
print("  ✓ Chat format: system/user/assistant (matches /api/chat)")
print("  ✓ System prompt: condensed constitution + intent instruction")
print("  ✓ All production fields: archetype, risk, causal, livability")
print("  ✓ Response format: <think> reasoning + structured response")
print("  ✓ 13 intents matching production IntentRouter")

print(f"\nOUTPUT FILES:")
for f in sorted(OUTPUT_DIR.glob("*")):
    size = f.stat().st_size / 1024**2 if f.is_file() else 0
    print(f"  - {f.name}" + (f" ({size:.1f} MB)" if size > 0 else ""))
print("=" * 60)

In [ ]:
# Cell 16: Deployment Guide
print("=" * 60)
print("  DEPLOYMENT GUIDE")
print("=" * 60)

print("\n1. DOWNLOAD FROM KAGGLE:")
print("   /kaggle/working/final_model/ → LoRA adapters")
print("   /kaggle/working/final_model/gguf/ → Q4_K_M GGUF")
print("   /kaggle/working/final_model/Modelfile.valora")

print("\n2. LOCAL OLLAMA SETUP:")
print("   # Copy GGUF + Modelfile to same directory, then:")
print("   ollama create valora-4b -f Modelfile.valora")
print("   ollama run valora-4b")

print("\n3. INTEGRATE WITH VALORA BACKEND:")
print("   # Update backend/.env:")
print("   OLLAMA_MODEL=valora-4b")
print("   # Or update via admin panel: /api/admin/llm-config")

print("\n4. TEST:")
print('   curl http://localhost:11434/api/generate -d \'{"model":"valora-4b","prompt":"Analyze Koramangala"}\'')

print("\n5. RESUME INTERRUPTED TRAINING:")
print("   1. Re-attach dataset")
print("   2. Run cells 1-11 (setup)")
print("   3. Run cell 12 (auto-resumes from checkpoint)")

print("\n" + "=" * 60)